In [ ]:
import os
import numpy as np
from PIL import Image
import tifffile
import tensorflow as tf

DATASET_BASE = '../data'         
OBJ           = 'leather'
MODEL_PATH    = 'checkpoints/ae_best.h5'       
OUT_BASE      = '../output/anomaly_maps'                
IMG_SIZE      = (224, 224)

model = tf.keras.models.load_model(MODEL_PATH)
normalize = tf.keras.layers.Rescaling(1./255)

# Creating anomaly maps


In [ ]:

test_root = os.path.join(DATASET_BASE, OBJ, 'test')
for defect in os.listdir(test_root):            # defect = 'good' or 'hole'
    src_dir = os.path.join(test_root, defect)
    dst_dir = os.path.join(OUT_BASE, OBJ, 'test', defect)
    os.makedirs(dst_dir, exist_ok=True)

    for fname in os.listdir(src_dir):
        if not fname.lower().endswith('.png'):
            continue

        # load & preprocess
        img = Image.open(os.path.join(src_dir, fname)).convert('RGB')
        img = img.resize(IMG_SIZE)
        arr = np.array(img, dtype=np.float32)   # H×W×3, [0–255]
        arr = arr[None, ...]  # add batch dimension
        arr = normalize(arr)                    # → [0–1]

        # reconstruct & compute per-pixel MSE
        rec = model(arr, training=False).numpy()[0]  
        amap = np.mean((arr[0] - rec)**2, axis=-1)  # H×W

        # save as float32 TIFF
        out_path = os.path.join(dst_dir, fname.replace('.png','') + '.tiff')
        tifffile.imwrite(out_path, amap.astype(np.float32))

print("Done generating anomaly maps.")


# Running eval function


In [ ]:
import subprocess

def run_leather_evaluation():
    cmd = [
        "python", "../evaluate/patchCore_Eval/evaluate_experiment2.py",
        "--dataset_base_dir", "../data",
        "--anomaly_maps_dir", "../output/anomaly_maps",
        "--output_dir", "../output/metrics",
        "--evaluated_objects", "leather",
        "--pro_integration_limit", "0.3"
    ]
    subprocess.run(cmd, check=True)

if __name__ == "__main__":
    run_leather_evaluation()
    print("Evaluation completed successfully.")